# Retrieval Evaluation — Python RAG Tutor

This notebook is a **documentation / proof artifact**: it runs the exact same evaluation as
`evaluation/evaluate_retrieval.py` (which the live project actually depends on — it writes
`data/best_config.json`, which `04_vector_representation.py` reads at startup) and shows the
full results table inline, so it can be reviewed on its own without opening a terminal.

It benchmarks **TF-IDF**, **BM25**, **embeddings-only**, and the **BM25+embeddings hybrid**
(swept from alpha=0.0 to alpha=1.0) against 136 hand-written ground-truth questions covering
all 68 topics in the knowledge base — using Precision@3, Recall@3, Hit Rate@3, and Mean
Reciprocal Rank, the same methodology used in Lab6/Lab7/Lab8.

**Run every cell top to bottom** to reproduce the results shown below with your own timestamp.

In [ ]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "evaluation" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "evaluation"))

from importlib import import_module

chunks = import_module("03_chunking").build_chunks()
ground_truth = import_module("ground_truth").ground_truth

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

print(f"Loaded {len(chunks)} chunks and {len(ground_truth)} ground-truth queries.")

## Metrics

Same four metrics used throughout the labs.

In [ ]:
K = 3

def precision_at_k(retrieved_ids, relevant_ids, k):
    hits = set(retrieved_ids[:k]).intersection(set(relevant_ids))
    return len(hits) / k

def recall_at_k(retrieved_ids, relevant_ids, k):
    hits = set(retrieved_ids[:k]).intersection(set(relevant_ids))
    return len(hits) / len(relevant_ids)

def hit_rate_at_k(retrieved_ids, relevant_ids, k):
    hits = set(retrieved_ids[:k]).intersection(set(relevant_ids))
    return int(len(hits) > 0)

def reciprocal_rank(retrieved_ids, relevant_ids):
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_ids:
            return 1 / rank
    return 0.0

def evaluate_retriever(name, retrieval_function, ground_truth, k=K):
    rows = []
    for item in ground_truth:
        query, relevant_ids = item["query"], [item["relevant_document_id"]]
        results = retrieval_function(query, k=max(k, 5))
        retrieved_ids = results["document_id"].tolist()
        rows.append({
            "retriever": name,
            "query": query,
            f"precision@{k}": precision_at_k(retrieved_ids, relevant_ids, k),
            f"recall@{k}": recall_at_k(retrieved_ids, relevant_ids, k),
            f"hit_rate@{k}": hit_rate_at_k(retrieved_ids, relevant_ids, k),
            "reciprocal_rank": reciprocal_rank(retrieved_ids, relevant_ids),
        })
    return pd.DataFrame(rows)

## Build the indices

TF-IDF, BM25, and sentence-embeddings, all over the same 68 chunks.

In [ ]:
def normalize_lexical_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\-]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def min_max_normalize(scores):
    scores = np.array(scores, dtype=float)
    if scores.max() == scores.min():
        return np.zeros_like(scores)
    return (scores - scores.min()) / (scores.max() - scores.min())

search_texts = [c["search_text"] for c in chunks]
doc_ids = [c["document_id"] for c in chunks]
titles = [c["title"] for c in chunks]

tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2))
tfidf_matrix = tfidf_vectorizer.fit_transform([normalize_lexical_text(t) for t in search_texts])

tokenized_chunks = [normalize_lexical_text(t).split() for t in search_texts]
bm25 = BM25Okapi(tokenized_chunks)

model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_embeddings = model.encode(search_texts, convert_to_numpy=True, normalize_embeddings=True)

print("Indices built.")

In [ ]:
def retrieve_top_k_tfidf(query, k=3):
    query_vector = tfidf_vectorizer.transform([normalize_lexical_text(query)])
    scores = cosine_similarity(query_vector, tfidf_matrix).flatten()
    ranking = np.argsort(scores)[::-1][:k]
    return pd.DataFrame({"document_id": [doc_ids[i] for i in ranking], "title": [titles[i] for i in ranking], "score": scores[ranking]})

def retrieve_top_k_hybrid(query, alpha, k=3):
    bm25_scores = bm25.get_scores(normalize_lexical_text(query).split())
    query_embedding = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    semantic_scores = cosine_similarity(query_embedding, chunk_embeddings).flatten()
    combined = (1 - alpha) * min_max_normalize(bm25_scores) + alpha * min_max_normalize(semantic_scores)
    ranking = np.argsort(combined)[::-1][:k]
    return pd.DataFrame({"document_id": [doc_ids[i] for i in ranking], "title": [titles[i] for i in ranking], "score": combined[ranking]})

## Run the comparison

TF-IDF and BM25-only/embeddings-only (the two edges of the hybrid formula) as baselines, then a full alpha sweep.

In [ ]:
ALPHA_SWEEP = [round(a, 1) for a in np.arange(0.1, 1.0, 0.1)]

tfidf_eval = evaluate_retriever("TF-IDF", lambda q, k: retrieve_top_k_tfidf(q, k), ground_truth)
bm25_eval = evaluate_retriever("BM25 (alpha=0.0)", lambda q, k: retrieve_top_k_hybrid(q, 0.0, k), ground_truth)
semantic_eval = evaluate_retriever("Embeddings (alpha=1.0)", lambda q, k: retrieve_top_k_hybrid(q, 1.0, k), ground_truth)

hybrid_evals = [
    evaluate_retriever(f"Hybrid alpha={a}", lambda q, k, a=a: retrieve_top_k_hybrid(q, a, k), ground_truth)
    for a in ALPHA_SWEEP
]

all_eval = pd.concat([tfidf_eval, bm25_eval, semantic_eval] + hybrid_evals, ignore_index=True)

summary = (
    all_eval.groupby("retriever")[[f"precision@{K}", f"recall@{K}", f"hit_rate@{K}", "reciprocal_rank"]]
    .mean()
    .sort_values(by=["reciprocal_rank", f"hit_rate@{K}"], ascending=False)
)

summary.round(3)

## Result

The row at the top of the table above is the best-performing configuration by mean reciprocal
rank (tie-broken by Hit Rate@3). **This confirms which retriever the live project actually
uses**: `evaluation/evaluate_retrieval.py` runs this identical sweep and writes the winning
alpha to `data/best_config.json`, which `04_vector_representation.py` reads automatically at
startup — so the production app always uses whichever configuration this notebook shows is
best, no manual code editing required.

This notebook is the readable proof of that decision; `evaluate_retrieval.py` is the script
the project depends on to actually apply it.